# Kaggle runner (thin)
All logic lives in the repo. This notebook only: pulls the repo, installs deps, runs **one** script.

Settings: Accelerator **GPU T4 x2**, Internet **On**. Add secrets (Add-ons -> Secrets): `HF_TOKEN`, and `GH_TOKEN` if the repo is private.
To persist outputs: **Save Version** (Save & Run All), then use this notebook's output as a Dataset input in later runs.

In [ ]:
# ---- edit per run ----
REPO = "github.com/<your-username>/mcp-poison-detect"
BRANCH = "main"
SCRIPT = "scripts/check_env.py"          # e.g. scripts/train_detector.py
ARGS = "--config configs/default.yaml"   # e.g. --set detector.use_context=false --tag b3
GPU_DEPS = False                          # True for attacker / agent-eval runs (installs vLLM)

In [ ]:
import os, subprocess
from kaggle_secrets import UserSecretsClient
sec = UserSecretsClient()
def secret(k):
    try: return sec.get_secret(k)
    except Exception: return None
os.environ["HF_TOKEN"] = secret("HF_TOKEN") or ""
gh = secret("GH_TOKEN")
url = f"https://{gh}@{REPO}.git" if gh else f"https://{REPO}.git"
os.chdir("/kaggle/working")
if os.path.isdir("mcp-poison-detect/.git"):
    subprocess.run(f"git -C mcp-poison-detect fetch -q && git -C mcp-poison-detect checkout -q {BRANCH} && git -C mcp-poison-detect pull -q", shell=True, check=True)
else:
    subprocess.run(f"git clone -q -b {BRANCH} {url} mcp-poison-detect", shell=True, check=True)
os.chdir("/kaggle/working/mcp-poison-detect")
!git log -1 --oneline

In [ ]:
!pip install -q -r requirements.txt
if GPU_DEPS:
    !pip install -q -r requirements-gpu.txt
!pip install -q -e . --no-deps

In [ ]:
# Resuming a previous run: attach its output as an input Dataset, then copy it in, e.g.
# !cp -r /kaggle/input/<prev-output>/outputs /kaggle/working/
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python {SCRIPT} {ARGS}